# Football AI System - Quick Start Demo

This notebook demonstrates the Football AI system with configurable, modern football video analysis.

## 1. Setup and Import

In [1]:
import sys
sys.path.append('/workspaces/football_analysis')

from football_ai import FootballAnalysisPipeline, get_broadcast_config

print("✅ Football AI system imported successfully!")

✅ Football AI system imported successfully!


## 2. Configuration Setup

The system provides several pre-configured setups for different use cases:

In [2]:
# Use broadcast quality configuration for professional output
config = get_broadcast_config()

# Update paths for this demo
config.update_paths(
    model_path="/workspaces/football_analysis/models/detect/best.pt",
    input_video_path="/workspaces/football_analysis/input_videos/08fd33_4.mp4",
    output_video_path="/workspaces/football_analysis/outputs/videos/demo_output.mp4",
    output_directory="/workspaces/football_analysis/outputs/data"
)

# Optional: Customize specific settings
config.model.confidence_threshold = 0.6
config.rendering.show_speeds = True

print("📋 Configuration Summary:")
print(f"Player Model: {config.model.player_model_path.split('/')[-1]}")
print(f"Keypoint Model: {config.model.field_model_path.split('/')[-1] if config.model.enable_keypoint_detection else 'Disabled'}")
print(f"Detection confidence: {config.model.confidence_threshold}")
print(f"Show tracks: {config.rendering.show_tracks}")
print(f"Show speeds: {config.rendering.show_speeds}")
print(f"Output directory: {config.processing.output_directory}")

# NEW: Demonstrate field dimension presets
print("\n🏟️ Field Dimension Presets Available:")
from football_ai.constants import FieldDimensions

# Show some popular presets
popular_presets = ['FIFA_STANDARD', 'MLS', 'YOUTH_U12', 'FIVE_A_SIDE']
for preset in popular_presets:
    info = getattr(FieldDimensions, preset)
    print(f"  {preset}: {info['width']}m x {info['height']}m - {info['description']}")

print(f"\n💡 This demo uses FIFA standard field dimensions (68m x 105m)")

📋 Configuration Summary:
Player Model: best.pt
Keypoint Model: best.pt
Detection confidence: 0.6
Show tracks: True
Show speeds: True
Output directory: /workspaces/football_analysis/outputs/data

🏟️ Field Dimension Presets Available:
  FIFA_STANDARD: 68.0m x 105.0m - Official FIFA standard dimensions for international matches
  MLS: 70.0m x 110.0m - Major League Soccer standard dimensions
  YOUTH_U12: 45.0m x 64.0m - Youth field for players under 12
  FIVE_A_SIDE: 25.0m x 42.0m - Standard 5-a-side field dimensions

💡 This demo uses FIFA standard field dimensions (68m x 105m)


## 2.5 Field Dimension Presets

The system now supports various predefined field dimensions for different types of soccer:

In [3]:
from football_ai.transformation.coordinate_transformer import PerspectiveCoordinateTransformer

print("🏟️ Creating coordinate transformers with different field presets:")
print()

# Example 1: FIFA Standard (default)
fifa_transformer = PerspectiveCoordinateTransformer.for_fifa_standard()
print(f"FIFA Standard: {fifa_transformer.field_width}m x {fifa_transformer.field_height}m")

# Example 2: Youth soccer
youth_transformer = PerspectiveCoordinateTransformer.for_youth('U12')
print(f"Youth U12: {youth_transformer.field_width}m x {youth_transformer.field_height}m")

# Example 3: Specific league (MLS has different dimensions)
mls_transformer = PerspectiveCoordinateTransformer.for_league('mls')
print(f"MLS: {mls_transformer.field_width}m x {mls_transformer.field_height}m")

# Example 4: Small-sided games
five_a_side_transformer = PerspectiveCoordinateTransformer.for_small_sided('5-a-side')
print(f"5-a-side: {five_a_side_transformer.field_width}m x {five_a_side_transformer.field_height}m")

# Example 5: Custom dimensions
custom_transformer = PerspectiveCoordinateTransformer(field_width=75.0, field_height=110.0)
print(f"Custom field: {custom_transformer.field_width}m x {custom_transformer.field_height}m")

print("\n💡 All transformers work the same way - just set field corners and start transforming!")
print("   transformer.set_field_corners(detected_corners)")
print("   field_coord = transformer.transform_point(pixel_coord)")

🏟️ Creating coordinate transformers with different field presets:

FIFA Standard: 68.0m x 105.0m
Youth U12: 45.0m x 64.0m
MLS: 70.0m x 110.0m
5-a-side: 25.0m x 42.0m
Custom field: 75.0m x 110.0m

💡 All transformers work the same way - just set field corners and start transforming!
   transformer.set_field_corners(detected_corners)
   field_coord = transformer.transform_point(pixel_coord)


## 3. Initialize Pipeline

Create the analysis pipeline with your configuration:

In [4]:
# Create pipeline with configuration
pipeline = FootballAnalysisPipeline(config=config)

print("⚽ Football analysis pipeline ready!")
print(f"Using player model: {pipeline.config.model.player_model_path.split('/')[-1]}")
print(f"Using keypoint model: {pipeline.config.model.field_model_path.split('/')[-1] if pipeline.config.model.enable_keypoint_detection else 'Disabled'}")
print(f"Output directory: {pipeline.config.processing.output_directory}")

Coordinate transformer calibrated successfully
Football AI analysis pipeline initialized successfully!
⚽ Football analysis pipeline ready!
Using player model: best.pt
Using keypoint model: best.pt
Output directory: /workspaces/football_analysis/outputs/data


## 4. Process Video

Run the complete football analysis:

In [5]:

print("🎬 Starting video analysis...")
print(f"Input: {config.processing.input_video_path.split('/')[-1]}")
print(f"Output: {config.processing.output_video_path.split('/')[-1]}")

try:
    # Process the video
    results = pipeline.process_video(
        video_path=config.processing.input_video_path,
        output_video_path=config.processing.output_video_path,
    )
    
    print("\n🎉 Analysis Complete!")
    print(f"📊 Frames processed: {results.total_frames}")
    
    # Count entities by type
    from football_ai.domain.models import FieldEntityType
    
    player_count = sum(1 for track_id, states in results.field_entity_tracks.items() 
                      if states and states[0].entity_type in [FieldEntityType.PLAYER, FieldEntityType.GOALKEEPER])
    referee_count = sum(1 for track_id, states in results.field_entity_tracks.items() 
                       if states and states[0].entity_type == FieldEntityType.REFEREE)
    
    print(f"👥 Players tracked: {player_count}")
    print(f"👨‍⚽ Referees tracked: {referee_count}")
    print(f"⚽ Ball tracking frames: {len(results.ball_tracks)}")
    
    # Team possession
    if len(results.team_ball_control) >= 2:
        print(f"\n🏟️ Team Possession:")
        print(f"   Team 1: {results.team_ball_control[0]:.1f}%")
        print(f"   Team 2: {results.team_ball_control[1]:.1f}%")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

🎬 Starting video analysis...
Input: 08fd33_4.mp4
Output: demo_output.mp4
Starting analysis of video: /workspaces/football_analysis/input_videos/08fd33_4.mp4
Setting up video processing environment...
Video properties: {'fps': 25.0, 'frame_count': 750, 'width': 1920, 'height': 1080, 'duration': 30.0}
Using configured field keypoints
Output video will be saved to: /workspaces/football_analysis/outputs/videos/demo_output.mp4
Starting frame-by-frame processing...


Processing frames:   0%|          | 1/750 [00:01<22:53,  1.83s/it]

Analyzing team colors with 18 players at frame 0
Team colors detected:
  Team 0: BGR(128, 197, 145)
  Team 1: BGR(230, 229, 220)
  Separation distance: 130.1
Team colors analyzed and assigned 18 players
Assignment stats: {'total_assigned': 18, 'team_1': 14, 'team_2': 3, 'unknown': 1}
Assigned 2 new players to teams


Processing frames:   2%|▏         | 18/750 [00:02<00:48, 15.08it/s]

Assigned 1 new players to teams
Assigned 1 new players to teams
Assigned 1 new players to teams


...

Assigned 1 new players to teams
Assigned 1 new players to teams


Processing frames:  98%|█████████▊| 732/750 [00:15<00:00, 51.46it/s]

Assigned 1 new players to teams
Assigned 1 new players to teams


Processing frames: 100%|██████████| 750/750 [00:16<00:00, 46.66it/s]

Assigned 1 new players to teams
Processed 750 frames successfully
Generating final analysis results...
Analysis complete: 56 entity tracks, 0 ball tracks, 750 frames
Caching disabled, skipping cache save
Video analysis completed successfully

🎉 Analysis Complete!
📊 Frames processed: 750
👥 Players tracked: 56
👨‍⚽ Referees tracked: 0
⚽ Ball tracking frames: 0

🏟️ Team Possession:
   Team 1: 100.0%
   Team 2: 0.0%


## 5. Results Summary

View key statistics from the analysis:

In [6]:
if 'results' in locals():
    print("📈 Analysis Summary:")
    print(f"   📹 Video: {results.total_frames} frames at {results.fps} FPS")
    print(f"   ⏱️ Duration: {results.total_frames/results.fps:.1f} seconds")
    
    # Filter field entities by type
    from football_ai.domain.models import FieldEntityType
    
    # Count entities by type
    player_tracks = {}
    referee_tracks = {}
    
    for track_id, entity_states in results.field_entity_tracks.items():
        if entity_states:  # Check if there are states for this track
            entity_type = entity_states[0].entity_type
            if entity_type in [FieldEntityType.PLAYER, FieldEntityType.GOALKEEPER]:
                player_tracks[track_id] = entity_states
            elif entity_type == FieldEntityType.REFEREE:
                referee_tracks[track_id] = entity_states
    
    print(f"   👥 Players: {len(player_tracks)} unique tracks")
    print(f"   👨‍⚽ Referees: {len(referee_tracks)} unique tracks")
    print(f"   ⚽ Ball tracking frames: {len(results.ball_tracks)}")
    
    # Show top 3 most active players
    player_activity = [(tid, len(states)) for tid, states in player_tracks.items()]
    player_activity.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n🏃 Most Active Players:")
    for i, (player_id, frame_count) in enumerate(player_activity[:3]):
        print(f"   {i+1}. Player {player_id}: {frame_count} frames ({frame_count/results.total_frames*100:.1f}% of video)")
    
    print(f"\n📁 Output Files:")
    print(f"   🎬 Annotated video: {config.processing.output_video_path}")
    print(f"   💾 Analysis data: {config.processing.output_directory}/analysis_cache.pkl")
    
    print(f"\n⚙️ Configuration Used:")
    print(f"   🎯 Detection confidence: {config.model.confidence_threshold}")
    print(f"   📊 Tracking threshold: {config.tracking.track_threshold}")
    print(f"   🎨 Rendering: {'Professional' if config.rendering.show_tracks else 'Basic'}")
else:
    print("⚠️ No results available. Please run the video analysis first.")

📈 Analysis Summary:
   📹 Video: 750 frames at 25.0 FPS
   ⏱️ Duration: 30.0 seconds
   👥 Players: 56 unique tracks
   👨‍⚽ Referees: 0 unique tracks
   ⚽ Ball tracking frames: 0

🏃 Most Active Players:
   1. Player 5: 741 frames (98.8% of video)
   2. Player 12: 741 frames (98.8% of video)
   3. Player 13: 741 frames (98.8% of video)

📁 Output Files:
   🎬 Annotated video: /workspaces/football_analysis/outputs/videos/demo_output.mp4
   💾 Analysis data: /workspaces/football_analysis/outputs/data/analysis_cache.pkl

⚙️ Configuration Used:
   🎯 Detection confidence: 0.6
   📊 Tracking threshold: 0.6
   🎨 Rendering: Professional


## Configuration Options

The Football AI system offers flexible configuration options:

### Quick Start Options:
- `get_default_config()` - Balanced performance and accuracy
- `get_high_accuracy_config()` - Maximum analysis quality
- `get_fast_processing_config()` - Speed optimized
- `get_broadcast_config()` - Professional broadcast quality

### Custom Configuration:
```python
from football_ai import FootballAIConfig

config = FootballAIConfig()
config.model.confidence_threshold = 0.7
config.rendering.show_speeds = True
config.save_to_file("my_config.json")
```

### Key Features:
- ✅ **Type-safe configuration** with validation
- ✅ **JSON save/load** for reproducibility
- ✅ **Hierarchical settings** (model, tracking, rendering, etc.)
- ✅ **Backward compatibility** with legacy parameters
- ✅ **Professional visualization** with customizable colors and overlays

**Ready to analyze your football videos with modern AI! 🚀**

In [7]:
# Quick test to verify the pipeline works after fixing the TeamColor/TeamFeatures issue
print("Testing pipeline initialization...")
from football_ai import FootballAnalysisPipeline, get_default_config

config = get_default_config()
test_pipeline = FootballAnalysisPipeline(config)
print("✅ Pipeline initialization successful!")
print("Ready to process videos.")

Testing pipeline initialization...
Coordinate transformer calibrated successfully
Football AI analysis pipeline initialized successfully!
✅ Pipeline initialization successful!
Ready to process videos.


In [8]:
# Test video analysis with the fixed TeamColor/TeamFeatures issue
print("🎬 Testing video analysis with fixed pipeline...")

# Use the existing pipeline variable
input_video = "/workspaces/football_analysis/input_videos/08fd33_4.mp4"
output_video = "/workspaces/football_analysis/outputs/videos/demo_output_test.mp4"

print(f"Input: {input_video}")
print(f"Output: {output_video}")

try:
    # Run just a few frames to test the fix
    original_max_frames = pipeline.config.processing.max_frames_to_process
    pipeline.config.processing.max_frames_to_process = 50  # Just process 50 frames for testing
    
    results = pipeline.process_video(
        video_path=input_video,
        output_video_path=output_video
    )
    
    print("✅ Video analysis completed successfully!")
    print(f"Total frames processed: {results.total_frames}")
    print(f"Field entity tracks: {len(results.field_entity_tracks)}")
    
    # Restore original setting
    pipeline.config.processing.max_frames_to_process = original_max_frames
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

🎬 Testing video analysis with fixed pipeline...
Input: /workspaces/football_analysis/input_videos/08fd33_4.mp4
Output: /workspaces/football_analysis/outputs/videos/demo_output_test.mp4
Starting analysis of video: /workspaces/football_analysis/input_videos/08fd33_4.mp4
Setting up video processing environment...
Video properties: {'fps': 25.0, 'frame_count': 750, 'width': 1920, 'height': 1080, 'duration': 30.0}
Using configured field keypoints
Output video will be saved to: /workspaces/football_analysis/outputs/videos/demo_output_test.mp4
Starting frame-by-frame processing...


Processing frames:   0%|          | 3/750 [00:00<00:28, 26.51it/s]

Analyzing team colors with 18 players at frame 0
Team colors detected:
  Team 0: BGR(230, 229, 220)
  Team 1: BGR(128, 197, 145)
  Separation distance: 130.1
Team colors analyzed and assigned 1 players
Assignment stats: {'total_assigned': 56, 'team_1': 26, 'team_2': 12, 'unknown': 18}
Assigned 17 new players to teams
Assigned 2 new players to teams


Processing frames:   1%|          | 9/750 [00:00<00:17, 42.87it/s]

Assigned 1 new players to teams


...

Assigned 1 new players to teams
Assigned 1 new players to teams


Processing frames:  98%|█████████▊| 734/750 [00:14<00:00, 51.15it/s]

Assigned 1 new players to teams
Assigned 1 new players to teams


Processing frames: 100%|██████████| 750/750 [00:14<00:00, 51.96it/s]


Assigned 1 new players to teams
Processed 750 frames successfully
Generating final analysis results...
Analysis complete: 56 entity tracks, 0 ball tracks, 750 frames
Caching disabled, skipping cache save
Video analysis completed successfully
✅ Video analysis completed successfully!
Total frames processed: 750
Field entity tracks: 56


In [9]:
# Reload modules to ensure we're using the updated code
import importlib
import sys

# Reload the modules we changed
if 'football_ai.analysis.team_color_analyzer' in sys.modules:
    importlib.reload(sys.modules['football_ai.analysis.team_color_analyzer'])
if 'football_ai.pipeline' in sys.modules:
    importlib.reload(sys.modules['football_ai.pipeline'])
if 'football_ai.assignment.team_assigner' in sys.modules:
    importlib.reload(sys.modules['football_ai.assignment.team_assigner'])

print("✅ Modules reloaded")

# Create a fresh pipeline instance
from football_ai import FootballAnalysisPipeline, get_default_config
fresh_config = get_default_config()
fresh_pipeline = FootballAnalysisPipeline(fresh_config)
print("✅ Fresh pipeline created")

✅ Modules reloaded
Coordinate transformer calibrated successfully
Football AI analysis pipeline initialized successfully!
✅ Fresh pipeline created


In [10]:
# Test video analysis with fresh pipeline after module reload
print("🎬 Testing video analysis with fresh reloaded pipeline...")

input_video = "/workspaces/football_analysis/input_videos/08fd33_4.mp4"
output_video = "/workspaces/football_analysis/outputs/videos/demo_output_test2.mp4"

print(f"Input: {input_video}")
print(f"Output: {output_video}")

try:
    # Run just a few frames to test the fix
    fresh_pipeline.config.processing.max_frames_to_process = 50  # Just process 50 frames for testing
    
    results = fresh_pipeline.process_video(
        video_path=input_video,
        output_video_path=output_video
    )
    
    print("✅ Video analysis completed successfully!")
    print(f"Total frames processed: {results.total_frames}")
    print(f"Field entity tracks: {len(results.field_entity_tracks)}")
    
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()

🎬 Testing video analysis with fresh reloaded pipeline...
Input: /workspaces/football_analysis/input_videos/08fd33_4.mp4
Output: /workspaces/football_analysis/outputs/videos/demo_output_test2.mp4
Starting analysis of video: /workspaces/football_analysis/input_videos/08fd33_4.mp4
Setting up video processing environment...
Video properties: {'fps': 25.0, 'frame_count': 750, 'width': 1920, 'height': 1080, 'duration': 30.0}
Using configured field keypoints
Output video will be saved to: /workspaces/football_analysis/outputs/videos/demo_output_test2.mp4
Starting frame-by-frame processing...


Processing frames:   1%|          | 7/750 [00:00<01:13, 10.16it/s]

Analyzing team colors with 19 players at frame 0
Team colors detected:
  Team 0: BGR(126, 193, 149)
  Team 1: BGR(230, 229, 220)
  Separation distance: 130.4
Team colors analyzed and assigned 19 players
Assignment stats: {'total_assigned': 19, 'team_1': 14, 'team_2': 3, 'unknown': 2}
Assigned 1 new players to teams
Assigned 1 new players to teams
Assigned 2 new players to teams


Processing frames:   3%|▎         | 19/750 [00:01<00:27, 27.01it/s]

Assigned 1 new players to teams
Assigned 1 new players to teams


...

Assigned 1 new players to teams
Assigned 1 new players to teams


Processing frames:  97%|█████████▋| 728/750 [00:14<00:00, 51.11it/s]

Assigned 1 new players to teams
Assigned 1 new players to teams
Assigned 1 new players to teams
Assigned 1 new players to teams


Processing frames: 100%|██████████| 750/750 [00:15<00:00, 49.70it/s]


Processed 750 frames successfully
Generating final analysis results...
Analysis complete: 62 entity tracks, 0 ball tracks, 750 frames
Caching disabled, skipping cache save
Video analysis completed successfully
✅ Video analysis completed successfully!
Total frames processed: 750
Field entity tracks: 62
